In [ ]:
# Cell 1: Install dependencies
print("=== Cell 1: Install dependencies ===")
!pip install -q google-genai fastapi uvicorn python-dotenv pyngrok httpx

In [ ]:
# Cell 2: API key
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['GOOGLE_API_KEY'] = secrets.get_secret('GOOGLE_API_KEY')
print('=== Cell 2: API key loaded ===')
print('Key set:', bool(os.environ.get('GOOGLE_API_KEY')))

In [ ]:
# Cell 3: Clone Chispa repo
print("=== Cell 3: Clone repo ===")
import subprocess, os

repo_url = "https://github.com/khalenanasser/chispa.git"
branch = "main"
target_dir = "/kaggle/working/chispa"

if os.path.exists(target_dir):
    print(f"Repo already exists at {target_dir}, pulling latest...")
    result = subprocess.run(["git", "-C", target_dir, "pull"], capture_output=True, text=True)
else:
    result = subprocess.run(
        ["git", "clone", "--branch", branch, "--depth", "1", repo_url, target_dir],
        capture_output=True, text=True
    )

print(result.stdout or result.stderr)
os.chdir(target_dir)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Cell 4: Start FastAPI server
print("=== Cell 4: Start FastAPI server ===")
import subprocess, time, os

server_process = subprocess.Popen(
    ["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    cwd="/kaggle/working/chispa"
)
time.sleep(3)
print("FastAPI server started on port 8000.")

In [ ]:
# Cell 5: Health check
print("=== Cell 5: Health check ===")
import httpx

resp = httpx.get("http://localhost:8000/health")
print(f"Status: {resp.status_code}")
print(f"Response: {resp.json()}")

In [ ]:
# Cell 6: ngrok tunnel
print("=== Cell 6: Start ngrok tunnel ===")
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
import os

ngrok_token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
ngrok.set_auth_token(ngrok_token)

public_url = ngrok.connect(8000)
os.environ["CHISPA_PUBLIC_URL"] = public_url.public_url
print(f"✦ Chispa is live at: {public_url.public_url}")